# 3 · PM decisions: a suspicious model, readable rules, three models, fairness

**What you practise (from the decision-tree lecture):**
- **Activity C:** find the product failure (a column the model should never see).
- Turning a tree into **rules** dispatch can read, and spotting a rule you shouldn't trust.
- **LR vs KNN vs tree** on the same orders.
- Checking whether the model works **equally well for different groups**.
- Prediction ≠ cause, and writing your **recommendation** for Lina.

> **How this notebook works**
> - Run the cells **from top to bottom** with `Shift + Enter`.
> - ▶️ **Run** cells work as they are.
> - ✏️ **Your turn** cells contain `___` (three underscores). Replace each `___` with your answer, then run the cell.
> - ✅ **Check** cells tell you whether you got it right. If a check fails, read the hint, fix the cell above and run both again.
> - 🧠 **Think** cells ask a PM question. Double-click the answer cell below it and write 1–3 sentences.
> - If you see `NameError: name '___' is not defined`, a blank is still empty.
> - Stuck? The `solutions/` folder has the same notebook, solved.

## Step 0 · Load, prepare and split (same as notebook 2)

> In this notebook we **reuse the test orders** to illustrate several PM checks. That's fine for learning; in a real project each new decision would be made on validation data, and the test set used once at the very end.

In [ ]:
# ▶️ Run: load the tools and the data, and prepare the inputs exactly like notebook 1
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import roc_auc_score, ConfusionMatrixDisplay

orders = pd.read_csv("data/quickcart_orders.csv")
features = ["order_hour", "weekday", "distance_km", "items_count", "basket_value_eur", "weather",
            "traffic_level", "store_queue", "courier_experience_months", "vehicle",
            "is_premium_customer", "promo_used", "customer_previous_orders"]
X = pd.get_dummies(orders[features], dtype=int)
y = orders["late"]
print("X:", X.shape, "| late share:", round(y.mean(), 3))

In [ ]:
# ▶️ Run: the same honest split as notebook 2 (75% to learn, 25% kept aside as the final exam)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("train:", X_train.shape[0], "orders | test:", X_test.shape[0], "orders, of which late:", y_test.sum())

## Step 1 · Activity C: Jonas's "amazing" model

Jonas, the data scientist, says: *"I used **all** the columns, including `order_id` and `customer_rating`. Look at this score!"*

In [ ]:
# ▶️ Run: Jonas's model (same split, same depth 4, but two extra columns)
X_jonas = X.assign(order_id=orders["order_id"], customer_rating=orders["customer_rating"])
Xj_train, Xj_test = X_jonas.loc[X_train.index], X_jonas.loc[X_test.index]
jonas_tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xj_train, y_train)
jonas_auc = roc_auc_score(y_test, jonas_tree.predict_proba(Xj_test)[:, 1])
print("Jonas's test AUC:", round(jonas_auc, 3), "  (our depth-4 tree: 0.770)")

In [ ]:
# ▶️ Run: which columns does Jonas's tree rely on? (feature importance = share of the mixing reduction)
importance = pd.Series(jonas_tree.feature_importances_, index=X_jonas.columns).sort_values(ascending=False)
importance.head(8).plot.barh(figsize=(7, 3.5), color="#0f766e").invert_yaxis()
plt.title("What Jonas's tree uses"); plt.xlabel("importance"); plt.show()
importance.head(5).round(3)

In [ ]:
# ✏️ Your turn: which column is suspicious? (write the column name)
suspicious_column = "___"

In [ ]:
# ✅ Check
assert suspicious_column.strip() == "customer_rating", "Look at the top bar."
print("✅ customer_rating does 94% of the work. Average rating: late orders",
      round(orders.loc[orders.late == 1, "customer_rating"].mean(), 2), "| on-time orders",
      round(orders.loc[orders.late == 0, "customer_rating"].mean(), 2))

🧠 **Think:** *when* does QuickCart know the customer's rating? Can the model use it when an order is placed? What is this mistake called?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

In [ ]:
# ✏️ Your turn: retrain WITHOUT the leaky column. Which column do we drop?
honest_train = Xj_train.drop(columns=["___"])
honest_test = Xj_test.drop(columns=["___"])
honest_tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(honest_train, y_train)
honest_auc = roc_auc_score(y_test, honest_tree.predict_proba(honest_test)[:, 1])
print("Test AUC without the leak:", round(honest_auc, 3))
print("Importance of order_id:", round(pd.Series(honest_tree.feature_importances_, index=honest_train.columns)["order_id"], 3))

🧠 **Think:** `order_id` got a small importance. Should an order number be a model input? Why or why not?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 2 · Rules dispatch can read

A depth-3 tree has at most 8 routes. `export_text` prints them. `weights: [on time, late]` = how many training orders ended there.

In [ ]:
# ▶️ Run: the rules of a depth-3 tree
rule_tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print(export_text(rule_tree, feature_names=list(X_train.columns), show_weights=True))

In [ ]:
# ▶️ Run: the same depth-3 tree as a picture. Find the tiny leaf on the far right (samples = 3)
plt.figure(figsize=(24, 9))
plot_tree(rule_tree, feature_names=list(X_train.columns), class_names=["on time", "late"],
          filled=True, rounded=True, fontsize=10)
plt.title("Depth-3 tree: 8 routes = 8 rules")
plt.show()

In [ ]:
# ▶️ Run: every leaf, sorted by late share (leaf numbers are scikit-learn's internal IDs)
leaves = pd.DataFrame({"leaf": rule_tree.apply(X_train), "late": y_train.values})
leaves.groupby("leaf")["late"].agg(late_share="mean", orders="size").sort_values("late_share", ascending=False).round(3)

In [ ]:
# ✏️ Your turn: the riskiest leaf has a late share of about 82% (71 orders).
# Read its route in the rules above and fill in the three numbers/words.
riskiest_rule = "distance_km > ___ AND store_queue > ___ AND basket_value_eur <= ___"
print(riskiest_rule)

In [ ]:
# ✅ Check
assert "___" not in riskiest_rule, "There is still a blank in your rule."
assert all(n in riskiest_rule for n in ["6.05", "10.5", "93.85"]), "Read the numbers on the route to the leaf with 58 late / 13 on time."
print("✅ Correct: far away AND busy store AND basket up to €93.85 → 82% of 71 past orders were late.")

🧠 **Think:** one leaf says *"far away, busy store, basket above €93.85 → 0% late"*. Look at how many orders it is based on. Would you put this rule on the dispatch screen? Which knob prevents it?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 3 · LR vs KNN vs tree on the same 750 orders

Logistic regression and KNN need **scaled** inputs; `make_pipeline` scales inside the model so we can't forget it.

**Fair comparison:** different models spread their scores differently, so the same threshold makes them flag different numbers of orders. We let **each model flag the same number of orders** (its riskiest 151, like our policy, and its riskiest 200, dispatch's maximum) and count how many late orders each one catches.

In [ ]:
# ✏️ Your turn: KNN looks at the 25 most similar past orders. Fill in the number of neighbours.
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=___)),
    "Decision tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=42),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    s = model.predict_proba(X_test)[:, 1]
    riskiest = np.argsort(-s, kind="stable")          # test orders sorted from highest to lowest score
    rows.append({"model": name, "test AUC": round(roc_auc_score(y_test, s), 3),
                 "late caught in top 151": y_test.values[riskiest[:151]].sum(),
                 "late caught in top 200": y_test.values[riskiest[:200]].sum()})
comparison = pd.DataFrame(rows)
comparison

🧠 **Think:** which model would you recommend to Lina, considering ranking quality (AUC), recall, and that dispatch must understand *why* an order is flagged?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 4 · Does the model work equally well for everyone?

A model can be good *on average* and still fail a group. Check **recall per group**: of the late orders in each group, how many did the policy flag? We look at our policy (threshold 0.4) and, for comparison, at 0.3.

In [ ]:
# ▶️ Run: attach the final tree's scores to the test orders
final_tree = models["Decision tree (depth 4)"]
test_orders = orders.loc[X_test.index].copy()
test_orders["score"] = final_tree.predict_proba(X_test)[:, 1]
test_orders["flag"] = (test_orders["score"] >= 0.4).astype(int)      # our policy from notebook 2
test_orders["flag_0.3"] = (test_orders["score"] >= 0.3).astype(int)  # a lower threshold, for comparison
test_orders[["distance_km", "weather", "vehicle", "late", "score", "flag", "flag_0.3"]].head()

In [ ]:
# ✏️ Your turn: recall per vehicle type. Which column do we group by?
late_only = test_orders[test_orders["late"] == 1]
late_only.groupby("___")[["flag", "flag_0.3"]].mean().round(2).assign(late_orders=late_only.groupby("___").size())

🧠 **Think:** the table shows recall per vehicle at our policy (`flag`, 0.4) and at 0.3. At 0.4 the groups look similar; at 0.3, late **bike** deliveries are caught less often (67%) than late **car** deliveries (78%). What does that teach you, and what would you check before launch? (Hint: also look at how many late orders each group has.)

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 5 · Prediction ≠ cause

The tree says: *far + bike → often late.* Lina's idea: **"Ban bikes for orders over 6 km. Problem solved!"**

🧠 **Think:** does the model prove that switching to a car prevents lateness? What evidence would you want first?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 6 · Your recommendation for Lina 📝

Write 5 short lines. Use numbers from notebooks 2 and 3.

1. **Recommendation** (ship / pilot / improve):
2. **Model and threshold:**
3. **What it achieves** (one business number):
4. **Biggest risk:**
5. **What we measure in the pilot:**

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

### 🎉 You finished all three notebooks!

You can now read a tree, explain how it chooses its questions, control its size, choose a threshold as a policy, spot a leak, and make a PM recommendation. Bring your recommendation to the stand-up.